# Scan4Resource API on Google Colab

Runs the FastAPI backend for the Scan4Resource door-scanner frontend and gives it a public **HTTPS** address, so your
frontend on Vercel (or on your phone during development) can call it.

**What you get:** `POST /detect` (find doors, count each once, size in cm, material) and `/scans` (save, list,
delete). It is the API of the merged Scan4Resource frontend.

**Before you start**
1. Use a **GPU** runtime (a T4 is fine). It also runs on CPU, but slowly.
2. Run the cells top to bottom. The first start downloads the models (a few minutes).
3. At the end you get a URL like `https://something.trycloudflare.com`. Put it in the frontend as `VITE_API_URL`.

**Good to know**
- The URL changes every time you run the tunnel cell, and the frontend bakes it in at build time. This setup is for
  development; see the README in the zip for a stable deployment.
- Saved scans are kept in a file on this Colab machine and disappear when the runtime resets.
- Sizes are estimates. Calibrate against a tape measure (see the calibration cell near the end).

## 0. Check you are on the Colab server

Run this first. If you open the notebook in VS Code, choose **Select Kernel → Colab → New Colab Server** (pick a GPU). If it stops with an error, the notebook is running on your own computer, and the next cells would install packages there and fail.

In [1]:
import os, shutil, subprocess, sys

if not os.path.isdir("/content"):
    raise RuntimeError(
        "This notebook is running on your own computer, not on a Colab server (there is no /content folder).\n"
        "In VS Code: click Select Kernel (top right) -> Colab -> New Colab Server, choose a GPU, then run this cell again.")

print("Running on the Colab server:", sys.executable)
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
else:
    print("No GPU on this server: it will work but slowly. Remove the server and create one with a GPU.")


RuntimeError: This notebook is running on your own computer, not on a Colab server (there is no /content folder).
In VS Code: click Select Kernel (top right) -> Colab -> New Colab Server, choose a GPU, then run this cell again.

## 1. Install

In [ ]:
!pip install -q fastapi "uvicorn[standard]" python-multipart pillow ultralytics "transformers>=4.45" requests
# YOLO-World (the zero-shot door detector) needs OpenAI's CLIP. Ultralytics installs it on demand; doing it now avoids a surprise at start-up.
!pip install -q git+https://github.com/ultralytics/CLIP.git
!mkdir -p /content/scan4resource-backend/app /content/scan4resource-backend/scripts

## 2. The backend code

Each cell below writes one file to `/content/scan4resource-backend/`. They are the same files as in the zip; you can
edit them here and restart the server (step 3).

In [ ]:
%%writefile /content/scan4resource-backend/app/__init__.py
"""Scan4Resource API."""


In [ ]:
%%writefile /content/scan4resource-backend/app/config.py
"""Settings, read from environment variables so nothing is hard-coded."""
from __future__ import annotations

import os
from dataclasses import dataclass


def _get(name: str, default, cast=str):
    raw = os.environ.get(name)
    if raw is None or raw.strip() == "":
        return default
    raw = raw.strip()
    if cast is bool:
        return raw.lower() in {"1", "true", "yes", "on"}
    return cast(raw)


@dataclass(frozen=True)
class Settings:
    # --- which components to run -------------------------------------------------
    pipeline: str = "real"  # "real" = YOLO + depth + CLIP; "fake" = simulated doors (connectivity test)

    # --- door detector -----------------------------------------------------------
    detector_weights: str = ""  # path to a fine-tuned door model (.pt); empty = zero-shot YOLO-World
    world_model: str = "yolov8s-worldv2.pt"
    detect_conf: float = 0.35
    detect_iou: float = 0.5
    detect_imgsz: int = 640
    min_box_h: float = 0.20  # ignore boxes shorter than this fraction of the frame height
    min_box_w: float = 0.04
    device: str = ""  # "" = automatic (GPU if available)

    # --- tracking (count each door once) ------------------------------------------
    min_hits: int = 2  # a door is reported only after it was seen in this many frames
    max_gap_s: float = 2.0  # a door not seen for this long can no longer be matched by position
    match_iou: float = 0.25
    match_dist: float = 0.25  # max centre distance (fraction of the frame) for appearance matching
    reid: bool = False  # re-identify a door that left the view and returned (by colour); see README
    reid_sim: float = 0.85
    session_ttl_s: int = 7200

    # --- measurement --------------------------------------------------------------
    measure_mode: str = "depth"  # "depth" (metric depth model) or "reference" (assume a standard door height)
    depth_model: str = "depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf"
    focal_ratio: float = 0.75  # focal length / long side of the image (about 0.75 for a phone's main camera)
    depth_scale: float = 1.0  # multiplier to calibrate against a tape measure
    reference_height_cm: float = 205.0

    # --- material -----------------------------------------------------------------
    material_model: str = "openai/clip-vit-base-patch32"
    material_min_prob: float = 0.35
    material_max_obs: int = 6  # classify each door in at most this many frames

    # --- server ---------------------------------------------------------------------
    allowed_origins: tuple = ("*",)
    db_path: str = "data/scans.db"

    @classmethod
    def from_env(cls) -> "Settings":
        origins = tuple(o.strip() for o in _get("ALLOWED_ORIGINS", "*").split(",") if o.strip())
        return cls(
            pipeline=_get("PIPELINE", cls.pipeline).lower(),
            detector_weights=_get("DETECTOR_WEIGHTS", cls.detector_weights),
            world_model=_get("WORLD_MODEL", cls.world_model),
            detect_conf=_get("DETECT_CONF", cls.detect_conf, float),
            detect_iou=_get("DETECT_IOU", cls.detect_iou, float),
            detect_imgsz=_get("DETECT_IMGSZ", cls.detect_imgsz, int),
            min_box_h=_get("MIN_BOX_H", cls.min_box_h, float),
            min_box_w=_get("MIN_BOX_W", cls.min_box_w, float),
            device=_get("DEVICE", cls.device),
            min_hits=_get("MIN_HITS", cls.min_hits, int),
            max_gap_s=_get("MAX_GAP_S", cls.max_gap_s, float),
            match_iou=_get("MATCH_IOU", cls.match_iou, float),
            match_dist=_get("MATCH_DIST", cls.match_dist, float),
            reid=_get("REID", cls.reid, bool),
            reid_sim=_get("REID_SIM", cls.reid_sim, float),
            session_ttl_s=_get("SESSION_TTL_S", cls.session_ttl_s, int),
            measure_mode=_get("MEASURE_MODE", cls.measure_mode).lower(),
            depth_model=_get("DEPTH_MODEL", cls.depth_model),
            focal_ratio=_get("FOCAL_RATIO", cls.focal_ratio, float),
            depth_scale=_get("DEPTH_SCALE", cls.depth_scale, float),
            reference_height_cm=_get("REFERENCE_HEIGHT_CM", cls.reference_height_cm, float),
            material_model=_get("MATERIAL_MODEL", cls.material_model),
            material_min_prob=_get("MATERIAL_MIN_PROB", cls.material_min_prob, float),
            material_max_obs=_get("MATERIAL_MAX_OBS", cls.material_max_obs, int),
            allowed_origins=origins or ("*",),
            db_path=_get("DB_PATH", cls.db_path),
        )


In [ ]:
%%writefile /content/scan4resource-backend/app/types.py
"""Small shared types."""
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Optional

import numpy as np


@dataclass
class Box:
    """A box in normalised image coordinates (0 to 1), origin top left."""

    x: float
    y: float
    w: float
    h: float

    @property
    def x1(self) -> float:
        return self.x + self.w

    @property
    def y1(self) -> float:
        return self.y + self.h

    @property
    def center(self) -> tuple[float, float]:
        return (self.x + self.w / 2, self.y + self.h / 2)

    @property
    def area(self) -> float:
        return max(0.0, self.w) * max(0.0, self.h)

    def iou(self, other: "Box") -> float:
        ix0, iy0 = max(self.x, other.x), max(self.y, other.y)
        ix1, iy1 = min(self.x1, other.x1), min(self.y1, other.y1)
        inter = max(0.0, ix1 - ix0) * max(0.0, iy1 - iy0)
        union = self.area + other.area - inter
        return inter / union if union > 0 else 0.0

    def center_distance(self, other: "Box") -> float:
        (ax, ay), (bx, by) = self.center, other.center
        return math.hypot(ax - bx, ay - by)

    def as_dict(self) -> dict:
        return {"x": round(self.x, 4), "y": round(self.y, 4), "w": round(self.w, 4), "h": round(self.h, 4)}


@dataclass
class Detection:
    box: Box
    conf: float
    hist: Optional[np.ndarray] = None  # colour histogram of the door, used to tell doors apart


@dataclass
class Observation:
    """One size reading of a door from one frame."""

    width_cm: float
    height_cm: float
    quality: float  # 0 to 1: how much to trust this reading


In [ ]:
%%writefile /content/scan4resource-backend/app/imaging.py
"""Image helpers: decoding uploads, cropping doors, colour histograms."""
from __future__ import annotations

import io
from typing import Optional

import numpy as np
from PIL import Image, ImageOps

from .types import Box

MAX_UPLOAD_BYTES = 8 * 1024 * 1024
MAX_SIDE = 1600
HIST_BINS = 8 * 4 * 4  # hue x saturation x value


def decode_image(data: bytes) -> Image.Image:
    """Decode an uploaded JPEG/PNG into an RGB image. Raises ValueError if it is not an image."""
    if not data:
        raise ValueError("The uploaded frame is empty.")
    if len(data) > MAX_UPLOAD_BYTES:
        raise ValueError("The uploaded frame is too large.")
    try:
        image = Image.open(io.BytesIO(data))
        image.load()
    except Exception as exc:  # Pillow raises several different exception types
        raise ValueError("The uploaded frame is not a valid image.") from exc
    image = ImageOps.exif_transpose(image).convert("RGB")
    if max(image.size) > MAX_SIDE:
        image.thumbnail((MAX_SIDE, MAX_SIDE))
    return image


def crop_box(image: Image.Image, box: Box, pad: float = 0.0) -> Image.Image:
    """Crop a normalised box out of the image. `pad` grows (positive) or shrinks (negative) it."""
    w, h = image.size
    x0 = max(0.0, box.x - pad * box.w)
    y0 = max(0.0, box.y - pad * box.h)
    x1 = min(1.0, box.x1 + pad * box.w)
    y1 = min(1.0, box.y1 + pad * box.h)
    left, top = int(round(x0 * w)), int(round(y0 * h))
    right, bottom = max(left + 1, int(round(x1 * w))), max(top + 1, int(round(y1 * h)))
    return image.crop((left, top, min(w, right), min(h, bottom)))


def color_histogram(crop: Image.Image) -> np.ndarray:
    """Normalised HSV histogram of a (small copy of a) crop. Cheap and good enough to tell doors apart."""
    small = crop.resize((24, 48)).convert("HSV")
    a = np.asarray(small, dtype=np.int32)
    idx = (a[..., 0] * 8 // 256) * 16 + (a[..., 1] * 4 // 256) * 4 + (a[..., 2] * 4 // 256)
    hist = np.bincount(idx.ravel(), minlength=HIST_BINS).astype(np.float32)
    return hist / max(1.0, float(hist.sum()))


def hist_similarity(a: Optional[np.ndarray], b: Optional[np.ndarray]) -> float:
    """Histogram intersection, 0 (nothing in common) to 1 (identical). Neutral 0.5 when unknown."""
    if a is None or b is None:
        return 0.5
    return float(np.minimum(a, b).sum())


In [ ]:
%%writefile /content/scan4resource-backend/app/tracker.py
"""Count each door once: give the same physical door the same id across frames.

Frames arrive about every 600 ms from a moving phone, so boxes move a lot between frames. A door is
matched to an existing track by overlap (IoU), or, if it moved further, by being close, similarly
sized and similarly coloured. Doors that leave the view and come back are, by default, counted as new:
identical white interior doors look alike, so matching them by colour would merge different doors
(see REID in the README).
"""
from __future__ import annotations

import math
import threading
import time
from typing import Optional

import numpy as np

from .config import Settings
from .imaging import hist_similarity
from .types import Box, Detection, Observation


def weighted_median(values: list[float], weights: list[float]) -> Optional[float]:
    if not values:
        return None
    order = np.argsort(values)
    v = np.asarray(values, dtype=float)[order]
    w = np.asarray(weights, dtype=float)[order]
    cum = np.cumsum(w)
    return float(v[int(np.searchsorted(cum, cum[-1] / 2.0))])


class Track:
    def __init__(self, track_id: str, det: Detection, now: float):
        self.id = track_id
        self.box = det.box
        self.conf = det.conf
        self.hist = det.hist
        self.hits = 1
        self.first_seen = now
        self.last_seen = now
        self._widths: list[float] = []
        self._heights: list[float] = []
        self._weights: list[float] = []
        self._material_sum: Optional[np.ndarray] = None
        self.material_obs = 0

    # ---- updates ------------------------------------------------------------------
    def update(self, det: Detection, now: float) -> None:
        self.box = det.box
        self.conf = 0.7 * self.conf + 0.3 * det.conf
        self.hits += 1
        self.last_seen = now
        if det.hist is not None:
            self.hist = det.hist if self.hist is None else 0.8 * self.hist + 0.2 * det.hist

    def add_observation(self, obs: Optional[Observation]) -> None:
        if obs is None or obs.quality <= 0:
            return
        self._widths.append(obs.width_cm)
        self._heights.append(obs.height_cm)
        self._weights.append(obs.quality)

    def add_material(self, probs: np.ndarray, weight: float) -> None:
        weighted = np.asarray(probs, dtype=float) * max(weight, 1e-3)
        self._material_sum = weighted if self._material_sum is None else self._material_sum + weighted
        self.material_obs += 1

    # ---- read-outs ------------------------------------------------------------------
    def confirmed(self, min_hits: int) -> bool:
        return self.hits >= min_hits

    def size_cm(self) -> tuple[Optional[float], Optional[float]]:
        w = weighted_median(self._widths, self._weights)
        h = weighted_median(self._heights, self._weights)
        return (round(w, 1) if w is not None else None, round(h, 1) if h is not None else None)

    def material_probs(self) -> Optional[np.ndarray]:
        if self._material_sum is None:
            return None
        return self._material_sum / float(self._material_sum.sum())

    def confidence(self) -> float:
        """Grows as more good size readings accumulate, so the app keeps the best (latest) reading."""
        reliability = 1.0 - math.exp(-sum(self._weights) / 3.0) if self._weights else 0.0
        return round(min(0.99, self.conf * (0.5 + 0.5 * reliability)), 3)


class SessionTracker:
    """All the doors seen during one walkthrough."""

    def __init__(self, cfg: Settings):
        self.cfg = cfg
        self.tracks: list[Track] = []
        self._next = 1
        self.lock = threading.Lock()

    def update(self, detections: list[Detection], now: float) -> list[tuple[Track, Detection]]:
        cfg = self.cfg
        active = [t for t in self.tracks if now - t.last_seen <= cfg.max_gap_s]

        candidates = []
        for di, det in enumerate(detections):
            for ti, track in enumerate(active):
                iou = det.box.iou(track.box)
                sim = hist_similarity(det.hist, track.hist)
                close = det.box.center_distance(track.box) <= cfg.match_dist and _similar_size(det.box, track.box)
                if iou >= cfg.match_iou or (close and sim >= 0.6):
                    candidates.append((iou + 0.5 * sim, di, ti))
        candidates.sort(reverse=True)

        used_det, used_track = set(), set()
        matched: list[tuple[Track, Detection]] = []
        for _, di, ti in candidates:
            if di in used_det or ti in used_track:
                continue
            used_det.add(di)
            used_track.add(ti)
            active[ti].update(detections[di], now)
            matched.append((active[ti], detections[di]))

        for di, det in enumerate(detections):
            if di in used_det:
                continue
            track = self._revive(det, now) if cfg.reid else None
            if track is not None:
                track.update(det, now)
            else:
                track = Track(f"door-{self._next}", det, now)
                self._next += 1
                self.tracks.append(track)
            matched.append((track, det))
        return matched

    def _revive(self, det: Detection, now: float) -> Optional[Track]:
        """Optional: match a returning door to an earlier one by colour and shape."""
        best, best_sim = None, self.cfg.reid_sim
        for track in self.tracks:
            if now - track.last_seen <= self.cfg.max_gap_s or not track.confirmed(self.cfg.min_hits):
                continue
            sim = hist_similarity(det.hist, track.hist)
            if sim >= best_sim and _similar_aspect(det.box, track.box):
                best, best_sim = track, sim
        return best

    @property
    def confirmed_count(self) -> int:
        return sum(1 for t in self.tracks if t.confirmed(self.cfg.min_hits))


def _similar_size(a: Box, b: Box) -> bool:
    ratio = a.h / b.h if b.h > 0 else 0.0
    return 0.6 <= ratio <= 1.7


def _similar_aspect(a: Box, b: Box) -> bool:
    if a.h <= 0 or b.h <= 0 or a.w <= 0 or b.w <= 0:
        return False
    return abs(math.log((a.w / a.h) / (b.w / b.h))) < 0.25


class TrackerStore:
    """One SessionTracker per scan id, with old sessions dropped after a while."""

    def __init__(self, cfg: Settings):
        self.cfg = cfg
        self._sessions: dict[str, tuple[SessionTracker, float]] = {}
        self._lock = threading.Lock()

    def get(self, session_id: str, now: Optional[float] = None) -> SessionTracker:
        now = time.monotonic() if now is None else now
        with self._lock:
            for key in [k for k, (_, seen) in self._sessions.items() if now - seen > self.cfg.session_ttl_s]:
                del self._sessions[key]
            tracker = self._sessions[session_id][0] if session_id in self._sessions else SessionTracker(self.cfg)
            self._sessions[session_id] = (tracker, now)
            return tracker


In [ ]:
%%writefile /content/scan4resource-backend/app/measure.py
"""Door size in centimetres from one frame.

depth mode      A metric depth model gives the distance to the door; with the camera's focal length the
                door's edges in the image become real-world points, and width/height are the distances
                between them (so a door seen at an angle is still measured correctly).
reference mode  No depth: assumes the door is a standard height and takes the width from the box shape.
                Rough, but needs no model.

Both are estimates from a phone camera, not survey-grade. Calibrate DEPTH_SCALE against a tape measure.
"""
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Optional

import numpy as np
from PIL import Image

from .config import Settings
from .types import Box, Observation

# A reading outside these ranges is not a door (or is a bad depth estimate): ignore it.
WIDTH_RANGE_CM = (45.0, 230.0)
HEIGHT_RANGE_CM = (160.0, 300.0)
MIN_FRONTAL = 0.6  # cosine of the angle between the camera axis and the door's normal


@dataclass
class FrameContext:
    width: int
    height: int
    depth: Optional[np.ndarray] = None  # metres, same size as the frame


def _strip_median(depth: np.ndarray, x0: float, x1: float, y0: float, y1: float) -> Optional[float]:
    h, w = depth.shape
    xi0, xi1 = int(max(0, math.floor(x0))), int(min(w, math.ceil(x1)))
    yi0, yi1 = int(max(0, math.floor(y0))), int(min(h, math.ceil(y1)))
    if xi1 - xi0 < 1 or yi1 - yi0 < 1:
        return None
    patch = depth[yi0:yi1, xi0:xi1]
    patch = patch[np.isfinite(patch) & (patch > 0.2)]
    if patch.size < 4:
        return None
    return float(np.median(patch))


def _extrapolate_inverse(z_a: float, p_a: float, z_b: float, p_b: float, at: float) -> float:
    """Depth at pixel position `at`, given depths at two positions along a row or column.

    For a flat surface, 1/depth changes linearly across the image, so extending that line to the edge of
    the door is exact for a plane and only mildly noise-sensitive.
    """
    slope = (1.0 / z_b - 1.0 / z_a) / (p_b - p_a)
    inv = 1.0 / z_a + slope * (at - p_a)
    return 1.0 / max(inv, 1e-3)


def size_from_depth(depth: np.ndarray, box: Box, f_px: float, scale: float = 1.0) -> Optional[Observation]:
    """Width and height of the door in `box`, or None if this frame cannot give a trustworthy reading."""
    H, W = depth.shape
    x0, y0, x1, y1 = box.x * W, box.y * H, box.x1 * W, box.y1 * H
    bw, bh = x1 - x0, y1 - y0
    if bw < 8 or bh < 16:
        return None
    # A door cut off by the edge of the frame would be measured too small.
    if x0 < 0.01 * W or y0 < 0.01 * H or x1 > 0.99 * W or y1 > 0.99 * H:
        return None

    inset = 0.08  # sample just inside the box edges, where the door surface is
    sw, sh = max(2.0, 0.04 * bw), max(2.0, 0.04 * bh)
    ym = (y0 + y1) / 2
    xa, xb = x0 + inset * bw, x1 - inset * bw  # left / right sampling columns
    ya, yb_ = y0 + inset * bh, y1 - inset * bh  # top / bottom sampling rows

    # Width: depth at the left and right edges (measured just inside, then extended to the edge).
    zl = _strip_median(depth, xa - sw, xa + sw, ym - 0.2 * bh, ym + 0.2 * bh)
    zr = _strip_median(depth, xb - sw, xb + sw, ym - 0.2 * bh, ym + 0.2 * bh)
    if zl is None or zr is None:
        return None
    zl_edge = _extrapolate_inverse(zl, xa, zr, xb, x0)
    zr_edge = _extrapolate_inverse(zl, xa, zr, xb, x1)

    cx, cy = W / 2, H / 2
    p_left = np.array([(x0 - cx) * zl_edge / f_px, zl_edge])
    p_right = np.array([(x1 - cx) * zr_edge / f_px, zr_edge])
    width_m = float(np.linalg.norm(p_right - p_left))
    frontal = math.cos(math.atan2(abs(p_right[1] - p_left[1]), max(abs(p_right[0] - p_left[0]), 1e-6)))
    if frontal < MIN_FRONTAL:
        return None

    # Height: on the edge nearest the camera, because that is the edge that sets the box's top and bottom
    # when the door is turned. Measure top and bottom there, extended to the box's top and bottom.
    near_col, u_near = (xa, x0) if zl_edge <= zr_edge else (xb, x1)
    zt = _strip_median(depth, near_col - sw, near_col + sw, ya - sh, ya + sh)
    zb = _strip_median(depth, near_col - sw, near_col + sw, yb_ - sh, yb_ + sh)
    if zt is None or zb is None:
        return None
    zt_edge = _extrapolate_inverse(zt, ya, zb, yb_, y0)
    zb_edge = _extrapolate_inverse(zt, ya, zb, yb_, y1)
    p_top = np.array([(u_near - cx) * zt_edge / f_px, (y0 - cy) * zt_edge / f_px, zt_edge])
    p_bottom = np.array([(u_near - cx) * zb_edge / f_px, (y1 - cy) * zb_edge / f_px, zb_edge])
    height_m = float(np.linalg.norm(p_bottom - p_top))

    width_cm, height_cm = width_m * 100 * scale, height_m * 100 * scale
    if not (WIDTH_RANGE_CM[0] <= width_cm <= WIDTH_RANGE_CM[1] and HEIGHT_RANGE_CM[0] <= height_cm <= HEIGHT_RANGE_CM[1]):
        return None

    quality = frontal**2 * min(1.0, 0.35 + bh / H)  # bigger, more frontal doors give better readings
    return Observation(width_cm, height_cm, quality)


def size_from_reference(box: Box, frame_w: int, frame_h: int, reference_height_cm: float) -> Optional[Observation]:
    """Assume a standard height and scale the width by the box's proportions."""
    if box.x < 0.01 or box.y < 0.01 or box.x1 > 0.99 or box.y1 > 0.99:
        return None
    bw_px, bh_px = box.w * frame_w, box.h * frame_h
    if bh_px < 16 or bw_px < 8:
        return None
    return Observation(reference_height_cm * bw_px / bh_px, reference_height_cm, 0.3)


# --------------------------------------------------------------------------- measurers
class ReferenceMeasurer:
    """No model needed."""

    name = "reference"

    def __init__(self, cfg: Settings):
        self.cfg = cfg

    def prepare(self, image: Image.Image) -> FrameContext:
        return FrameContext(image.width, image.height)

    def measure(self, ctx: FrameContext, box: Box) -> Optional[Observation]:
        return size_from_reference(box, ctx.width, ctx.height, self.cfg.reference_height_cm)


class DepthMeasurer:
    """Metric depth with Depth Anything V2 (indoor model, outputs metres)."""

    name = "depth"

    def __init__(self, cfg: Settings):
        import torch
        from transformers import AutoImageProcessor, AutoModelForDepthEstimation

        self.cfg = cfg
        self.torch = torch
        self.device = cfg.device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.processor = AutoImageProcessor.from_pretrained(cfg.depth_model)
        self.model = AutoModelForDepthEstimation.from_pretrained(cfg.depth_model).to(self.device).eval()

    def prepare(self, image: Image.Image) -> FrameContext:
        torch = self.torch
        inputs = self.processor(images=image, return_tensors="pt").to(self.device)
        with torch.no_grad():
            predicted = self.model(**inputs).predicted_depth  # (1, h, w) in metres
            depth = torch.nn.functional.interpolate(
                predicted.unsqueeze(1), size=(image.height, image.width), mode="bicubic", align_corners=False
            )
        return FrameContext(image.width, image.height, depth[0, 0].float().cpu().numpy())

    def measure(self, ctx: FrameContext, box: Box) -> Optional[Observation]:
        if ctx.depth is None:
            return None
        f_px = self.cfg.focal_ratio * max(ctx.width, ctx.height)
        return size_from_depth(ctx.depth, box, f_px, self.cfg.depth_scale)


In [ ]:
%%writefile /content/scan4resource-backend/app/material.py
"""Door material from a photo of the door, with CLIP zero-shot classification (no training data needed)."""
from __future__ import annotations

from typing import Optional

import numpy as np
from PIL import Image

from .config import Settings

MATERIALS = ["wood", "metal", "glass", "pvc", "composite"]

PROMPTS = {
    "wood": [
        "a photo of a wooden door",
        "a door made of natural wood with visible wood grain",
        "a painted wooden interior door",
    ],
    "metal": ["a photo of a metal door", "a steel door", "an aluminium door"],
    "glass": ["a photo of a glass door", "a door made mostly of glass panels", "a glass sliding door"],
    "pvc": ["a photo of a white plastic PVC door", "a uPVC door"],
    "composite": ["a photo of a composite door", "a fibreglass door", "a laminate door"],
}


def _features(out, torch):
    """CLIP feature calls return a tensor in most transformers versions; newer ones may wrap it."""
    if isinstance(out, torch.Tensor):
        return out
    for attr in ("image_embeds", "text_embeds", "pooler_output"):
        value = getattr(out, attr, None)
        if value is not None:
            return value
    return out[0]


def pick_material(probs: Optional[np.ndarray], min_prob: float) -> str:
    """The most likely material, or 'unknown' when the classifier is not confident."""
    if probs is None:
        return "unknown"
    i = int(np.argmax(probs))
    return MATERIALS[i] if float(probs[i]) >= min_prob else "unknown"


class ClipMaterialClassifier:
    def __init__(self, cfg: Settings):
        import torch
        from transformers import CLIPModel, CLIPProcessor

        self.torch = torch
        self.device = cfg.device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = CLIPModel.from_pretrained(cfg.material_model).to(self.device).eval()
        self.processor = CLIPProcessor.from_pretrained(cfg.material_model)

        rows = []
        for material in MATERIALS:
            tokens = self.processor(text=PROMPTS[material], return_tensors="pt", padding=True).to(self.device)
            with torch.no_grad():
                emb = _features(self.model.get_text_features(**tokens), torch)
            emb = emb / emb.norm(dim=-1, keepdim=True)
            mean = emb.mean(dim=0)
            rows.append(mean / mean.norm())
        self.text = torch.stack(rows)  # (materials, dim)

    def classify(self, crops: list[Image.Image]) -> np.ndarray:
        """(N, len(MATERIALS)) probabilities."""
        if not crops:
            return np.zeros((0, len(MATERIALS)))
        torch = self.torch
        inputs = self.processor(images=crops, return_tensors="pt").to(self.device)
        with torch.no_grad():
            emb = _features(self.model.get_image_features(**inputs), torch)
            emb = emb / emb.norm(dim=-1, keepdim=True)
            probs = (100.0 * emb @ self.text.T).softmax(dim=-1)
        return probs.cpu().numpy()


In [ ]:
%%writefile /content/scan4resource-backend/app/detector.py
"""Door detection with Ultralytics YOLO.

Default: YOLO-World, an open-vocabulary model, asked for "door" (works with no training, but is noisier).
Better: a YOLO model fine-tuned on doors (see scripts/train_door_detector.py). Point DETECTOR_WEIGHTS at it.
"""
from __future__ import annotations

from PIL import Image

from .config import Settings
from .types import Box, Detection

ACCEPTED_NAMES = {"door", "doors"}


class YoloDoorDetector:
    def __init__(self, cfg: Settings):
        from ultralytics import YOLO, YOLOWorld

        self.cfg = cfg
        self.class_ids = None
        if cfg.detector_weights:
            self.model = YOLO(cfg.detector_weights)
            names = self.model.names  # {id: name}
            ids = [i for i, n in names.items() if str(n).lower() in ACCEPTED_NAMES]
            # A model with a single, differently named class is assumed to be a door model.
            self.class_ids = ids or (None if len(names) == 1 else [])
            if self.class_ids == []:
                raise ValueError(f"None of the classes {list(names.values())} is called 'door'.")
            self.source = f"fine-tuned model {cfg.detector_weights}"
        else:
            self.model = YOLOWorld(cfg.world_model)
            self.model.set_classes(["door"])
            self.source = f"zero-shot {cfg.world_model}"

    def detect(self, image: Image.Image) -> list[Detection]:
        cfg = self.cfg
        results = self.model.predict(
            image,
            conf=cfg.detect_conf,
            iou=cfg.detect_iou,
            imgsz=cfg.detect_imgsz,
            device=cfg.device or None,
            classes=self.class_ids,
            verbose=False,
        )
        detections: list[Detection] = []
        if not results:
            return detections
        boxes = results[0].boxes
        if boxes is None or len(boxes) == 0:
            return detections
        xyxyn = boxes.xyxyn.cpu().numpy()
        confs = boxes.conf.cpu().numpy()
        for (x0, y0, x1, y1), conf in zip(xyxyn, confs):
            x0, y0, x1, y1 = max(0.0, float(x0)), max(0.0, float(y0)), min(1.0, float(x1)), min(1.0, float(y1))
            box = Box(x0, y0, x1 - x0, y1 - y0)
            if box.h >= cfg.min_box_h and box.w >= cfg.min_box_w:
                detections.append(Detection(box, float(conf)))
        return detections


In [ ]:
%%writefile /content/scan4resource-backend/app/fake.py
"""Stand-ins for the models, so the whole app can be tested without a GPU.

PIPELINE=fake runs these: doors drift into and out of view, then the loop repeats. It is meant for checking
that the frontend, the tunnel and the API are connected. It is NOT real detection.
"""
from __future__ import annotations

import numpy as np
from PIL import Image

from .material import MATERIALS
from .types import Box, Detection

_SCRIPT = [  # (first tick, last tick, box)
    (2, 9, Box(0.06, 0.14, 0.30, 0.72)),
    (12, 19, Box(0.62, 0.20, 0.30, 0.66)),
    (22, 29, Box(0.34, 0.12, 0.32, 0.76)),
]
_LOOP = 34


class FakeDetector:
    source = "simulated doors"

    def __init__(self):
        self.tick = 0

    def detect(self, image: Image.Image) -> list[Detection]:
        self.tick = (self.tick + 1) % _LOOP
        t = self.tick
        out = []
        for first, last, box in _SCRIPT:
            if first <= t <= last:
                drift = np.sin(t / 2) * 0.008
                out.append(Detection(Box(box.x + drift, box.y + drift / 2, box.w, box.h), 0.9))
        return out


class FakeMaterial:
    def classify(self, crops: list[Image.Image]) -> np.ndarray:
        probs = np.full((len(crops), len(MATERIALS)), 0.05)
        probs[:, 0] = 0.8  # wood
        return probs


In [ ]:
%%writefile /content/scan4resource-backend/app/pipeline.py
"""One frame in, the doors in it out: detect, track, measure, classify material."""
from __future__ import annotations

import threading
import time
from typing import Optional

from PIL import Image

from .config import Settings
from .imaging import color_histogram, crop_box
from .material import pick_material
from .tracker import TrackerStore
from .types import Detection


class Pipeline:
    def __init__(self, cfg: Settings, detector, measurer, material):
        self.cfg = cfg
        self.detector = detector
        self.measurer = measurer
        self.material = material
        self.trackers = TrackerStore(cfg)
        self._lock = threading.Lock()  # one frame at a time: the models are not thread-safe

    def describe(self) -> dict:
        return {
            "detector": getattr(self.detector, "source", type(self.detector).__name__),
            "measure_mode": getattr(self.measurer, "name", type(self.measurer).__name__),
            "material": type(self.material).__name__,
        }

    def process(self, image: Image.Image, session_id: str, now: Optional[float] = None) -> list[dict]:
        """The doors visible in this frame, each with a stable id and its best size/material so far."""
        now = time.monotonic() if now is None else now
        with self._lock:
            detections = self.detector.detect(image)
            for det in detections:
                det.hist = color_histogram(crop_box(image, det.box))

            tracker = self.trackers.get(session_id, now)
            with tracker.lock:
                matched = tracker.update(detections, now)
                if not matched:
                    return []

                ctx = self.measurer.prepare(image)
                to_classify: list[tuple] = []
                for track, det in matched:
                    track.add_observation(self.measurer.measure(ctx, det.box))
                    if track.material_obs < self.cfg.material_max_obs:
                        to_classify.append((track, det, crop_box(image, det.box, pad=-0.05)))

                if to_classify:
                    probs = self.material.classify([crop for _, _, crop in to_classify])
                    for (track, det, _), p in zip(to_classify, probs):
                        track.add_material(p, det.conf)

                return [self._view(track, det) for track, det in matched if track.confirmed(self.cfg.min_hits)]

    def _view(self, track, det: Detection) -> dict:
        width, height = track.size_cm()
        return {
            "id": track.id,
            "box": det.box.as_dict(),
            "width_cm": width,
            "height_cm": height,
            "material": pick_material(track.material_probs(), self.cfg.material_min_prob),
            "confidence": track.confidence(),
        }


def build_pipeline(cfg: Settings) -> Pipeline:
    """Create the components named by the settings. Models are downloaded on first use."""
    if cfg.pipeline == "fake":
        from .fake import FakeDetector, FakeMaterial
        from .measure import ReferenceMeasurer

        return Pipeline(cfg, FakeDetector(), ReferenceMeasurer(cfg), FakeMaterial())

    from .detector import YoloDoorDetector
    from .material import ClipMaterialClassifier

    detector = YoloDoorDetector(cfg)
    if cfg.measure_mode == "reference":
        from .measure import ReferenceMeasurer

        measurer = ReferenceMeasurer(cfg)
    else:
        from .measure import DepthMeasurer

        measurer = DepthMeasurer(cfg)
    return Pipeline(cfg, detector, measurer, ClipMaterialClassifier(cfg))


In [ ]:
%%writefile /content/scan4resource-backend/app/schemas.py
"""Request bodies for saving scans."""
from __future__ import annotations

from typing import Optional, Union

from pydantic import BaseModel, ConfigDict, Field, field_validator


class DoorIn(BaseModel):
    model_config = ConfigDict(extra="ignore")

    id: Optional[Union[str, int]] = None
    width_cm: Optional[float] = None
    height_cm: Optional[float] = None
    material: Optional[str] = None
    confidence: Optional[float] = None
    image: Optional[str] = None  # small JPEG thumbnail as a data URL (or null)

    @field_validator("id")
    @classmethod
    def _id_to_str(cls, value):
        return None if value is None else str(value)


class ScanIn(BaseModel):
    model_config = ConfigDict(extra="ignore")

    id: Optional[str] = None
    site: Optional[str] = None  # what the person typed as the site name (may be empty)
    started_at: Optional[str] = None
    finished_at: Optional[str] = None
    doors: list[DoorIn] = Field(default_factory=list)


In [ ]:
%%writefile /content/scan4resource-backend/app/store.py
"""Saved scans, in SQLite (a single file, no server to run)."""
from __future__ import annotations

import json
import sqlite3
import threading
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

from .schemas import ScanIn


class ScanStore:
    def __init__(self, path: str):
        if path != ":memory:":
            Path(path).expanduser().resolve().parent.mkdir(parents=True, exist_ok=True)
        self._db = sqlite3.connect(path, check_same_thread=False)
        self._db.row_factory = sqlite3.Row
        self._lock = threading.Lock()
        with self._lock:
            self._db.execute(
                """CREATE TABLE IF NOT EXISTS scans (
                       id TEXT PRIMARY KEY,
                       site TEXT NOT NULL,
                       started_at TEXT,
                       finished_at TEXT,
                       created_at TEXT NOT NULL,
                       doors TEXT NOT NULL)"""
            )
            self._migrate_old_layout()
            self._db.commit()

    def _migrate_old_layout(self) -> None:
        """Earlier versions of this API kept both a `name` and a `site` column. Keep one (`site`)."""
        columns = {row["name"] for row in self._db.execute("PRAGMA table_info(scans)")}
        if "name" not in columns:
            return
        self._db.execute("UPDATE scans SET site = name WHERE site = ''")
        try:
            self._db.execute("ALTER TABLE scans DROP COLUMN name")
        except sqlite3.OperationalError as exc:  # SQLite older than 3.35
            raise RuntimeError(
                "The scans database uses an old layout and this SQLite cannot upgrade it. "
                "Delete the database file (DB_PATH) to start fresh."
            ) from exc

    def save(self, scan: ScanIn) -> dict:
        record = {
            "id": scan.id or str(uuid.uuid4()),
            "site": (scan.site or "").strip(),
            "started_at": scan.started_at,
            "finished_at": scan.finished_at,
            "created_at": datetime.now(timezone.utc).isoformat(),
            "doors": [door.model_dump() for door in scan.doors],
        }
        with self._lock:  # saving the same id again replaces the earlier save (a retry does not duplicate)
            self._db.execute(
                "INSERT OR REPLACE INTO scans (id, site, started_at, finished_at, created_at, doors) "
                "VALUES (?, ?, ?, ?, ?, ?)",
                (
                    record["id"], record["site"], record["started_at"], record["finished_at"],
                    record["created_at"], json.dumps(record["doors"]),
                ),
            )
            self._db.commit()
        return record

    def list(self) -> list[dict]:
        with self._lock:
            rows = self._db.execute("SELECT * FROM scans ORDER BY created_at DESC").fetchall()
        return [self._to_dict(row) for row in rows]

    def get(self, scan_id: str) -> Optional[dict]:
        with self._lock:
            row = self._db.execute("SELECT * FROM scans WHERE id = ?", (scan_id,)).fetchone()
        return self._to_dict(row) if row else None

    def delete(self, scan_id: str) -> bool:
        with self._lock:
            cur = self._db.execute("DELETE FROM scans WHERE id = ?", (scan_id,))
            self._db.commit()
        return cur.rowcount > 0

    @staticmethod
    def _to_dict(row: sqlite3.Row) -> dict:
        data = dict(row)
        data["doors"] = json.loads(data["doors"])
        return data


In [ ]:
%%writefile /content/scan4resource-backend/app/main.py
"""Scan4Resource API, for the Scan4Resource frontend.

    POST   /detect       multipart: frame, scan_id, frame_index       -> { doors: [...] }
    POST   /scans        json: { id?, site?, started_at?, finished_at?, doors: [...] }
    GET    /scans        -> [ scan, ... ]  newest first
    DELETE /scans/{id}   -> 204, or 404
    GET    /health

Run:  uvicorn app.main:app --host 0.0.0.0 --port 8000
"""
from __future__ import annotations

import logging
from contextlib import asynccontextmanager
from typing import Optional

from fastapi import FastAPI, File, Form, HTTPException, Response, UploadFile
from fastapi.middleware.cors import CORSMiddleware

from .config import Settings
from .imaging import decode_image
from .pipeline import Pipeline, build_pipeline
from .schemas import ScanIn
from .store import ScanStore

log = logging.getLogger("scan4resource")


def create_app(settings: Optional[Settings] = None, pipeline: Optional[Pipeline] = None) -> FastAPI:
    settings = settings or Settings.from_env()

    @asynccontextmanager
    async def lifespan(app: FastAPI):
        app.state.store = ScanStore(settings.db_path)
        app.state.pipeline = pipeline or build_pipeline(settings)  # loads the models once, at start-up
        log.info("Pipeline ready: %s", app.state.pipeline.describe())
        yield

    app = FastAPI(title="Scan4Resource API", version="0.1.0", lifespan=lifespan)
    app.add_middleware(
        CORSMiddleware,
        allow_origins=list(settings.allowed_origins),
        allow_methods=["*"],
        allow_headers=["*"],
        allow_credentials=False,
    )

    # ---- health --------------------------------------------------------------------
    @app.get("/")
    def root():
        return {"service": "Scan4Resource API", "status": "ok"}

    @app.get("/health")
    def health():
        return {"status": "ok", "pipeline": settings.pipeline, **app.state.pipeline.describe()}

    # ---- live detection ---------------------------------------------------------------
    @app.post("/detect")
    def detect(
        frame: UploadFile = File(...),
        scan_id: Optional[str] = Form(None),
        frame_index: Optional[int] = Form(None),
    ):
        """One camera frame in, the doors visible in it out. `scan_id` ties the frames of one walkthrough
        together, so a door seen in many frames keeps one id. Materials are the keys wood, metal, glass, pvc,
        composite or unknown."""
        try:
            image = decode_image(frame.file.read())
        except ValueError as exc:
            raise HTTPException(status_code=400, detail=str(exc)) from exc
        return {"doors": app.state.pipeline.process(image, scan_id or "default")}

    # ---- saved scans --------------------------------------------------------------------
    @app.post("/scans", status_code=201)
    def save_scan(scan: ScanIn):
        return app.state.store.save(scan)

    @app.get("/scans")
    def list_scans():
        return app.state.store.list()

    @app.delete("/scans/{scan_id}", status_code=204)
    def delete_scan(scan_id: str):
        if not app.state.store.delete(scan_id):
            raise HTTPException(status_code=404, detail="Scan not found.")
        return Response(status_code=204)

    return app


# `uvicorn app.main:app` builds the app from environment variables.
app = create_app()


In [ ]:
%%writefile /content/scan4resource-backend/scripts/train_door_detector.py
"""Fine-tune a YOLO model to detect doors, using the public DoorDetect dataset.

DoorDetect: 1,213 images with boxes for door, handle, cabinet door and refrigerator door.
Training on all four classes (not just "door") teaches the model what is NOT a room door, which cuts
false positives on cupboards and fridges. The API only uses the "door" class.

    python scripts/train_door_detector.py --epochs 60
    DETECTOR_WEIGHTS=weights/door.pt uvicorn app.main:app

Needs a GPU to be quick (about 25 minutes for 60 epochs on a Colab T4). Check the dataset's licence before using the
resulting model commercially: its images come from Open Images and MCIndoor20000.
"""
from __future__ import annotations

import argparse
import random
import shutil
import subprocess
from pathlib import Path

DATASET_URL = "https://github.com/MiguelARD/DoorDetect-Dataset.git"
NAMES = {0: "door", 1: "handle", 2: "cabinet door", 3: "refrigerator door"}  # the dataset's own class ids


def make_splits(dataset_dir: Path, out_dir: Path, val_fraction: float = 0.1, seed: int = 0) -> Path:
    """Write train.txt / val.txt (absolute image paths) and the YOLO data file. Returns the data file."""
    images = sorted(p for p in (dataset_dir / "images").iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    images = [p for p in images if (dataset_dir / "labels" / f"{p.stem}.txt").exists()]
    if not images:
        raise SystemExit(f"No labelled images found in {dataset_dir}")

    random.Random(seed).shuffle(images)
    n_val = max(1, int(len(images) * val_fraction))
    val, train = images[:n_val], images[n_val:]

    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "train.txt").write_text("\n".join(str(p.resolve()) for p in train) + "\n")
    (out_dir / "val.txt").write_text("\n".join(str(p.resolve()) for p in val) + "\n")

    data_file = out_dir / "door.yaml"
    names = "\n".join(f"  {i}: {name}" for i, name in NAMES.items())
    data_file.write_text(f"path: {out_dir.resolve()}\ntrain: train.txt\nval: val.txt\nnames:\n{names}\n")
    print(f"{len(train)} training images, {len(val)} validation images -> {data_file}")
    return data_file


def find_best_weights(model) -> Path:
    """Where Ultralytics saved best.pt. It chooses the folder (newer versions put a relative `project` under
    runs/detect/), so ask the trainer instead of guessing the path."""
    trainer = getattr(model, "trainer", None)
    best = Path(trainer.best) if trainer is not None and getattr(trainer, "best", None) else None
    if best is not None and best.exists():
        return best
    found = sorted(Path("runs").rglob("weights/best.pt"), key=lambda p: p.stat().st_mtime)
    if found:
        return found[-1]
    raise SystemExit("Training finished, but best.pt was not found under runs/.")


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--dataset", default="data/DoorDetect-Dataset", help="where to clone / find the dataset")
    parser.add_argument("--base", default="yolov8s.pt", help="pretrained model to start from")
    parser.add_argument("--epochs", type=int, default=60)
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument("--batch", type=int, default=16)
    parser.add_argument("--output", default="weights/door.pt")
    args = parser.parse_args()

    dataset = Path(args.dataset)
    if not (dataset / "images").exists():
        dataset.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--depth", "1", DATASET_URL, str(dataset)], check=True)

    data_file = make_splits(dataset, Path("data/door_splits"))

    from ultralytics import YOLO

    model = YOLO(args.base)
    model.train(data=str(data_file), epochs=args.epochs, imgsz=args.imgsz, batch=args.batch,
                project="runs", name="door", exist_ok=True)

    best = find_best_weights(model)
    out = Path(args.output)
    out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(best, out)
    print(f"\nSaved {out} (copied from {best}). Use it with:  DETECTOR_WEIGHTS={out}")


if __name__ == "__main__":
    main()


## 3. Start the API

`start_server()` (re)starts the API and waits until it is ready. Settings are passed as keyword arguments, for example
`start_server(DEPTH_SCALE=1.05)`. The first call downloads the models, so allow a few minutes.

To test the connection **before** the models work, use `start_server(PIPELINE="fake")`: it returns simulated doors.
`GET /health` always says which pipeline is running.

In [ ]:
import os, re, subprocess, sys, time
import requests

BACKEND = "/content/scan4resource-backend"
_server = None


def _tail(path, n=60):
    try:
        return "".join(open(path).readlines()[-n:])
    except FileNotFoundError:
        return "(no log yet)"


def stop_server():
    global _server
    if _server is not None and _server.poll() is None:
        _server.terminate()
        try:
            _server.wait(timeout=15)
        except subprocess.TimeoutExpired:
            _server.kill()
    _server = None
    subprocess.run(["pkill", "-f", "uvicorn app.main:app"], check=False)  # leftovers from an earlier session


def start_server(timeout=600, **settings):
    """(Re)start the API. Settings are environment variables, e.g. start_server(PIPELINE="fake")."""
    global _server
    stop_server()
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "DB_PATH": f"{BACKEND}/data/scans.db",
           **{k: str(v) for k, v in settings.items()}}
    log = open("/content/server.log", "w")
    _server = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
        cwd=BACKEND, env=env, stdout=log, stderr=subprocess.STDOUT,
    )
    started = time.time()
    while time.time() - started < timeout:
        if _server.poll() is not None:
            raise RuntimeError("The server stopped while starting. Log:\n" + _tail("/content/server.log"))
        try:
            r = requests.get("http://127.0.0.1:8000/health", timeout=2)
            if r.ok:
                print("API is up:", r.json())
                return r.json()
        except requests.RequestException:
            pass
        time.sleep(2)
    raise TimeoutError("The server did not become ready in time. Log:\n" + _tail("/content/server.log"))


# Real detection. Use PIPELINE="fake" first if you only want to test the connection.
start_server()

## 4. Public HTTPS address

Downloads `cloudflared` and opens a free quick tunnel to the API. No account is needed.

In [ ]:
CLOUDFLARED = "/content/cloudflared"
_tunnel = None


def start_tunnel(timeout=90):
    global _tunnel
    if not os.path.exists(CLOUDFLARED):
        subprocess.run(["curl", "-fsSL", "-o", CLOUDFLARED,
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
        os.chmod(CLOUDFLARED, 0o755)
    if _tunnel is not None and _tunnel.poll() is None:
        _tunnel.terminate()

    log = open("/content/tunnel.log", "w")
    _tunnel = subprocess.Popen([CLOUDFLARED, "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                               stdout=log, stderr=subprocess.STDOUT)
    url, started = None, time.time()
    while url is None and time.time() - started < timeout:
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
        if match:
            url = match.group(0)
        else:
            time.sleep(1)
    if url is None:
        raise RuntimeError("No tunnel address appeared. Log:\n" + _tail("/content/tunnel.log"))

    for _ in range(30):  # the address can take a few seconds to start working
        try:
            if requests.get(url + "/health", timeout=5, headers={"User-Agent": "Mozilla/5.0"}).ok:
                break
        except requests.RequestException:
            pass
        time.sleep(2)
    else:
        print("The tunnel exists but is not answering yet. Wait a minute and open the URL + /health in a browser.")
    return url


PUBLIC_URL = start_tunnel()
print("\nPublic API address:", PUBLIC_URL)
print("Health check:      ", PUBLIC_URL + "/health")
print("\nIn the frontend, set:\n    VITE_API_URL=" + PUBLIC_URL)

## 5. Connect the frontend

- **Local development:** in the frontend folder, create `.env.local` containing `VITE_API_URL=<the address above>`, then run
  `npm run dev` (or `npm run dev:phone` to open it on your phone over HTTPS).
- **Vercel:** Project Settings → Environment Variables → `VITE_API_URL` = the address above (no trailing slash), then
  **redeploy**. Vite reads the variable at build time, so a change needs a new deployment.

If the app says it can't reach the scan service, open `<address>/health` in the phone's browser first.

## 6. (Optional) Try it on a photo

Upload a photo of a door to `/content/door.jpg` (Files panel on the left), then run this cell. It sends the same frame
three times, like the app would, so the door is confirmed and measured.

In [ ]:
TEST_IMAGE = "/content/door.jpg"

if os.path.exists(TEST_IMAGE):
    for i in range(3):
        with open(TEST_IMAGE, "rb") as f:
            r = requests.post("http://127.0.0.1:8000/detect", files={"frame": f},
                              data={"scan_id": "colab-test", "frame_index": str(i)}, timeout=120)
        print(i, r.status_code, r.json())
else:
    print(f"Upload a photo to {TEST_IMAGE} first.")

## 7. (Optional) Calibrate the sizes

Measure one real door with a tape, open it in the app, and compare. Then restart with a correction factor:

```
DEPTH_SCALE = tape measure / size the app shows        # e.g. 91 / 86 = 1.058
```
```python
start_server(DEPTH_SCALE=1.058)
```

If sizes are off in a way that depends on distance or on the phone, `FOCAL_RATIO` (default 0.75, a phone's main camera)
is the other setting to try. Check several doors before trusting the numbers.

## 8. (Optional) Train a better door detector

The default zero-shot detector works without training but makes more mistakes. This fine-tunes YOLO on the public
DoorDetect dataset (about 25 minutes on a T4; it has only ~580 labelled room doors, so treat it as a starting point).
Check the dataset's licence before commercial use.

In [ ]:
!cd /content/scan4resource-backend && python scripts/train_door_detector.py --epochs 60

In [ ]:
# Use the trained model. The public address stays the same.
start_server(DETECTOR_WEIGHTS=f"{BACKEND}/weights/door.pt")

## Logs

In [ ]:
print("--- API log ---")
print(_tail("/content/server.log", 40))